In [8]:
print(hash("name"))
print(hash("age"))
print(hash("name"))  # same as first?

-5445335068280424305
-2941962267572652695
-5445335068280424305


In [9]:
index = hash("name") % 10
print(index)

index = hash("age") % 10
print(index)

5
5


In [11]:
table = [None] * 10
def set_value(key,value):
    index = hash(key) % 10
    table[index] = (key,value)

def get_value(key):
    index = hash(key) % 10
    pair = table[index]
    if pair is None:
        return None
    return pair[1]

set_value("name", "Saugat")
set_value("age", 24)

print(get_value("name"))
print(get_value("age"))

24
24


Instead of one value per slot, each slot holds a list of (key, value) pairs. When two keys collide, they both go into the same slot's list. When you look up, you search that list for the right key.
This is called separate chaining.

In [13]:
table = [[] for _ in range(10)]

def set_value(key,value):
    index = hash(key)%10
    for pair in table[index]:
        if pair[0] == key:
            pair[1] = value
            return
    table[index].append([key,value])

def get_value(key):
    index = hash(key) % 10
    for pair in table[index]:
        if pair[0] == key:
            return pair[1]
    return None

set_value("name","Saugat")
set_value("age","24")

print(get_value("name"))
print(get_value("age"))
print(table)

Saugat
24
[[], [], [], [], [], [['name', 'Saugat'], ['age', '24']], [], [], [], []]


In [19]:
table = [[] for _ in range(10)]

def set_value(key,value):
    index = hash(key)%10
    for pair in table[index]:
        if pair[0] == key:
            pair[1] = value
            return
    table[index].append([key,value])

def get_value(key):
    index = hash(key) % 10
    for pair in table[index]:
        if pair[0] == key:
            return pair[1]
    return 

def delete_value(key):
    index = hash(key) % 10
    for pair in table[index]:
        if pair[0] == key:
            table[index].remove(pair)
            return



In [20]:
set_value("name", "Saugat")
set_value("age", 24)
delete_value("name")
print(get_value("name"))   # should print None
print(get_value("age"))    # should print 24

None
24


In [21]:
items = 7
slots = 10
load_factor = items / slots
print(load_factor)  # 0.7

0.7


In [23]:
table = [[] for _ in range(10)]
size = 10

def set_value(key,value):
    index = hash(key) % size
    for pair in table[index]:
        if pair[0] == key:
            pair[1]=value
            return
    table[index].append([key,value])

def get_value(key):
    index = hash(key) % size
    for pair in table[index]:
        if pair[0] == key:
            return pair[1]
    return None

def delete_value(key):
    index = hash(key) % size
    for pair in table[index]:
        if pair[0] == key:
            table[index].remove(pair)
            return

def resize():
    global table,size
    new_size = size * 2
    new_table = [[] for _ in range(new_size)]
    for bucket in table:
        for pair in bucket:
            key,value = pair
            new_index = hash(key)%new_size
            new_table[new_index].append([key,value])
    table = new_table
    size = new_size

set_value("name", "Saugat")
set_value("age", 24)
resize()
print(get_value("name"))
print(get_value("age"))
print(len(table))


Saugat
24
20


In [ ]:
class HashTable:
    # Uses separate chaining
    def __init__(self,size=10):
        self.size = size
        self.count = 0
        self.table = [[] for _ in range(self.size)]

    def _hash(self,key):
        return hash(key)%self.size

    def __setitem__(self,key,value):
        index = self._hash(key)
        for pair in self.table[index]:
            if pair[0]==key:
                pair[1] = value
                return
        self.table[index].append([key,value])
        self.count += 1

    def __getitem__(self,key):
        index = self._hash(key)
        for pair in self.table[index]:
            if pair[0] == key:
                return pair[1]
        return None
    
    def __delitem__(self,key):
        index = self._hash(key)
        for pair in self.table[index]:
            if pair[0] == key:
                self.table[index].remove(pair)
                self.count -= 1
                return
    
    def resize(self):
        new_size = self.size *2
        new_table = [[] for _ in range(new_size)]
        for bucket in self.table:
            for pair in bucket:
                key,value = pair
                new_index = hash(key)%new_size
                new_table[new_index].append([key,value])
        self.table = new_table
        self.size = new_size

ht = HashTable()
ht["name"] = "Saugat"
ht["age"] = 24
print(ht.count)  # should be 2
del ht["name"]
print(ht.count)  # should be 1

2
1


In [3]:
class HashTable:
    def __init__(self,size = 10):
        self.size = size
        self.count = 0
        self.table = [[] for _ in range(self.size)]
        self.order = []
    
    def _hash(self,key):
        return hash(key) % self.size
    
    def __setitem__(self,key,value):
        index = self._hash(key)
        for pair in self.table[index]:
            if pair[0] == key:
                pair[1] = value
                return
        self.table[index].append([key,value])
        self.count+=1
        self.order.append(key)
        self._check_resize()
    
    def __getitem__(self,key):
        index = self._hash(key)
        for pair in self.table[index]:
            if pair[0] == key:
                return pair[1]
        raise KeyError(key)
    
    def __delitem__(self,key):
        index = self._hash(key)
        for pair in self.table[index]:
            if pair[0] == key:
                self.table[index].remove(pair)
                self.count -= 1
                self.order.remove(key)
                self._check_resize()
                return
                
    def __eq__(self,other):
        if self.count != other.count:
            return False
        for bucket in self.table:
            for pair in bucket:
                if other[pair[0]]!=pair[1]:
                    return False
        return True
    
    def __contains__(self, key):
        index = self._hash(key)
        for pair in self.order:
            if pair[0] == key:
                return True
        return False

    def items(self):
        for key in self.order:
            yield key,self[key]
    def keys(self):
        for key in self.order:
            yield key
    def values(self):
        for key in self.order:
            yield[key]

    def get(self,key,default=None):
        try:
            return self[key]
        except KeyError:
            return default

    def update(self,dictionary):
        for key,value in dictionary.items():
            self[key]=value

    def pop(self,key,default=None):
        try:
            value = self[key]
            del self[key]
            return value
        except KeyError:
            return default
    
    def clear(self):
        self.table = [[] for _ in range(self.size)]
        self.count = 0
        self.order = []

    def setdefault(self,key,default=None):
        if key in self:
            return self[key]
        self[key] = default
        return default


    def _resize(self,new_size):
        new_table = [[] for _ in range(new_size)]
        for bucket in self.table:
            for pair in bucket:
                key,value = pair
                new_index = hash(key)%new_size
                new_table[new_index].append([key,value])
        self.table=new_table
        self.size = new_size

    def _check_resize(self):
        load_factor = self.count / self.size
        if load_factor >= 0.7:
            self._resize(self.size * 2)
        elif load_factor <= 0.2 and self.size > 10:
            self._resize(self.size//2)

    @classmethod
    def from_dict(cls,dictionary,size=None):
        if size is None:
            size=max(10,len(dictionary)*2)
        ht=cls(size)
        for key,value in dictionary.items():
            ht[key] = value
        return ht

    def __iter__(self):
        for key in self.order:
            yield key


    def __repr__(self):
        pairs = [f"{k!r}: {v!r}" for k, v in self.items()]
        return f"HashTable({'{' + ', '.join(pairs) + '}'})"

    def __str__(self):
        pairs = [f"{k!r}: {v!r}" for k, v in self.items()]
        return "{" + ", ".join(pairs) + "}"

    def __len__(self):
        return self.count

ht = HashTable.from_dict({"name": "Saugat", "age": 24, "city": "Kathmandu"})
print(ht)
print(len(ht))

{'name': 'Saugat', 'age': 24, 'city': 'Kathmandu'}
3
